# 00 · Tablas de entrada y efecto de las máscaras

Exploración para entender los datos antes de la validación formal (notebook 01):

1. Cargar cada tabla con las funciones de `src/data.py` y ver su estructura.
2. Ver cómo se relacionan y cómo queda la tabla unida.
3. Aplicar las máscaras conservando los valores originales al lado.
4. Medir el impacto: cuántos registros quedan válidos y por qué motivo se excluye el resto.

No escribe nada en disco.

## 0. Dependencias y configuración

In [1]:
import os
import sys
from pathlib import Path

CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config
from src.data import (
    MASK_REASONS,
    REJECTION_CODES,
    apply_masks,
    build_daily_prices,
    compute_launch,
    load_calendar,
    load_prices,
    load_sales,
    profile_frame,
    week_of_day,
)

pd.set_option("display.max_columns", 40)
cfg = load_validated_config()
RAW_DIR = Path(cfg["data"]["raw_dir"])
LAST_DAY = cfg["data"]["last_day"]

# Muestra para la tabla unida (solo exploración): una tienda y algunos artículos.
SAMPLE_STORE = None  # None = la primera tienda
SAMPLE_N_ITEMS = 3

## 1. Calendario

`load_calendar` devuelve el CSV tal cual. Una fila por día `d`; `wm_yr_wk` es la semana que
conecta con los precios.

In [2]:
calendar = load_calendar(cfg)
print(f"filas: {calendar.shape[0]:,} | columnas: {calendar.shape[1]} | celdas: {calendar.size:,} | "
      f"nulos explícitos: {int(calendar.isna().sum().sum()):,}")
calendar.head()

filas: 1,969 | columnas: 14 | celdas: 27,566 | nulos explícitos: 7,542


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [3]:
profile_frame(calendar)

,dtype,nulls,null_pct,unique
date,str,0,0.00,1969
wm_yr_wk,int64,0,0.00,282
weekday,str,0,0.00,7
wday,int64,0,0.00,7
month,int64,0,0.00,12
year,int64,0,0.00,6
d,str,0,0.00,1969
event_name_1,str,1807,91.77,30
event_type_1,str,1807,91.77,4
event_name_2,str,1964,99.75,4


In [4]:
calendar["event_type_1"].unique()

<ArrowStringArray>
[nan, 'Sporting', 'Cultural', 'National', 'Religious']
Length: 5, dtype: str

In [5]:
calendar["event_name_1"].unique()

<ArrowStringArray>
[                  nan,           'SuperBowl',       'ValentinesDay',
       'PresidentsDay',           'LentStart',           'LentWeek2',
       'StPatricksDay',           'Purim End',      'OrthodoxEaster',
          'Pesach End',       'Cinco De Mayo',        'Mother's day',
         'MemorialDay',      'NBAFinalsStart',        'NBAFinalsEnd',
        'Father's day',     'IndependenceDay',      'Ramadan starts',
         'Eid al-Fitr',            'LaborDay',         'ColumbusDay',
           'Halloween',           'EidAlAdha',         'VeteransDay',
        'Thanksgiving',           'Christmas',        'Chanukah End',
             'NewYear',   'OrthodoxChristmas', 'MartinLutherKingDay',
              'Easter']
Length: 31, dtype: str

In [6]:
calendar["event_name_2"].unique()

<ArrowStringArray>
[nan, 'Easter', 'Cinco De Mayo', 'OrthodoxEaster', 'Father's day']
Length: 5, dtype: str

In [7]:
calendar["event_type_2"].unique()

<ArrowStringArray>
[nan, 'Cultural', 'Religious']
Length: 3, dtype: str

Los nulos en `event_name_*` y `event_type_*` son esperables: la mayoría de los días no
tienen evento.

## 2. Ventas

El CSV original es ancho: una fila por tienda-artículo y una columna por día (`d_1` …).
Primero, cómo viene el archivo:

In [8]:
sales_head = pd.read_csv(RAW_DIR / cfg["data"]["sales_file"], nrows=5)
print("columnas del CSV:", sales_head.shape[1])
sales_head.iloc[:, :12]

columnas del CSV: 1947


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,d_6
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0


In [9]:
sales_head

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10,d_11,d_12,d_13,d_14,...,d_1922,d_1923,d_1924,d_1925,d_1926,d_1927,d_1928,d_1929,d_1930,d_1931,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,2,1,2,2,1,0,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,2,1,3,0,0,1,0,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,2,2,4,2,1,1,1,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,3,1,1,3,2,3,2,2,2,2,0,0,0,2,1,0,0,2,1,0


`load_sales` conserva solo `store_id`, `item_id` y los días. Devuelve dos objetos:

- `units`: arreglo NumPy (series × días), float32. La fila i es la serie `series_index.loc[i]`
  y la columna j es el día `d_{j+1}`.
- `series_index`: `store_id, item_id, csv_row` (fila original en el CSV), ordenado por tienda y
  artículo.

In [10]:
units, series_index = load_sales(cfg)
print(f"series (filas): {units.shape[0]:,} | días (columnas): {units.shape[1]:,} | "
      f"celdas: {units.size:,} | nulos explícitos: {int(np.isnan(units).sum()):,}")
series_index.head()

series (filas): 30,490 | días (columnas): 1,941 | celdas: 59,181,090 | nulos explícitos: 0


,store_id,item_id,csv_row
0,CA_1,FOODS_1_001,1612
1,CA_1,FOODS_1_002,1613
2,CA_1,FOODS_1_003,1614
3,CA_1,FOODS_1_004,1615
4,CA_1,FOODS_1_005,1616


In [11]:
day_cols = [f"d_{d}" for d in range(1, LAST_DAY + 1)]
# Vista de las primeras filas como DataFrame (sin copiar el arreglo completo).
series_index.head().join(pd.DataFrame(units[:5], columns=day_cols)).iloc[:, :12]

,store_id,item_id,csv_row,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9
0,CA_1,FOODS_1_001,1612,3.0,0.0,0.0,1.0,4.0,2.0,0.0,2.0,0.0
1,CA_1,FOODS_1_002,1613,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
2,CA_1,FOODS_1_003,1614,0.0,0.0,0.0,0.0,1.0,3.0,1.0,1.0,1.0
3,CA_1,FOODS_1_004,1615,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CA_1,FOODS_1_005,1616,3.0,9.0,3.0,3.0,0.0,2.0,1.0,2.0,1.0


In [12]:
pd.Series({
    "tiendas": series_index["store_id"].nunique(),
    "artículos": series_index["item_id"].nunique(),
    "series tienda-artículo": len(series_index),
    "ceros": int((units == 0).sum()),
    "% ceros": round(float((units == 0).mean() * 100), 2),
    "negativos": int((units < 0).sum()),
    "no enteros": int((np.isfinite(units) & (units != np.floor(units))).sum()),
}, name="ventas", dtype=object)

tiendas                         10
artículos                     3049
series tienda-artículo       30490
ceros                     40241819
% ceros                       68.0
negativos                        0
no enteros                       0
Name: ventas, dtype: object

## 3. Precios

`load_prices` devuelve el CSV tal cual más `csv_row`. Una fila por tienda-artículo-semana
**con** precio: si una semana no tiene precio, la fila no existe (el faltante es implícito).

In [13]:
prices = load_prices(cfg)
print(f"filas: {prices.shape[0]:,} | columnas: {prices.shape[1]} | celdas: {prices.size:,} | "
      f"nulos explícitos: {int(prices.isna().sum().sum()):,}")
prices.head()

filas: 6,841,121 | columnas: 5 | celdas: 34,205,605 | nulos explícitos: 0


,store_id,item_id,wm_yr_wk,sell_price,csv_row
0,CA_1,HOBBIES_1_001,11325,9.58,0
1,CA_1,HOBBIES_1_001,11326,9.58,1
2,CA_1,HOBBIES_1_001,11327,8.26,2
3,CA_1,HOBBIES_1_001,11328,8.26,3
4,CA_1,HOBBIES_1_001,11329,8.26,4


In [14]:
profile_frame(prices)

,dtype,nulls,null_pct,unique
store_id,str,0,0.0,10
item_id,str,0,0.0,3049
wm_yr_wk,int64,0,0.0,282
sell_price,float64,0,0.0,1048
csv_row,int64,0,0.0,6841121


In [15]:
key = ["store_id", "item_id", "wm_yr_wk"]
pd.Series({
    "claves duplicadas (tienda, artículo, semana)": int(prices.duplicated(key).sum()),
    "precios ≤ 0": int((prices["sell_price"] <= 0).sum()),
    "semanas con precio": prices["wm_yr_wk"].nunique(),
}, name="precios")

claves duplicadas (tienda, artículo, semana)      0
precios ≤ 0                                       0
semanas con precio                              282
Name: precios, dtype: int64

## 4. Relaciones entre tablas

```
ventas (store_id, item_id, d_j) ──d──▶ calendario (d → date, wm_yr_wk)
                │                                   │
                └──── (store_id, item_id, wm_yr_wk) ◀┘──▶ precios (sell_price)
```

- Ventas → calendario por el día `d`.
- Ventas + calendario → precios por tienda, artículo y semana.

Cobertura de las llaves:

In [16]:
sales_keys = series_index.set_index(["store_id", "item_id"]).index
price_keys = prices.drop_duplicates(["store_id", "item_id"]).set_index(["store_id", "item_id"]).index
cal_days = set(calendar["d"])
pd.Series({
    "series de ventas sin ningún precio": int((~sales_keys.isin(price_keys)).sum()),
    "series de precios sin ventas": int((~price_keys.isin(sales_keys)).sum()),
    "días de ventas sin fila en calendario": sum(d not in cal_days for d in day_cols),
    "semanas de precios sin días en calendario":
        int((~prices["wm_yr_wk"].isin(calendar["wm_yr_wk"])).sum()),
}, name="cobertura")

series de ventas sin ningún precio           0
series de precios sin ventas                 0
días de ventas sin fila en calendario        0
semanas de precios sin días en calendario    0
Name: cobertura, dtype: int64

### Tabla unida (muestra)

La tabla unida en formato largo tendría una fila por tienda-artículo-día (series × días; con
M5 son ~59 millones), así que se construye para una muestra. Un `sell_price` nulo aquí es una
semana **sin fila** en precios: el faltante implícito se vuelve explícito.

In [17]:
store = SAMPLE_STORE or series_index["store_id"].iloc[0]
rows = series_index.index[series_index["store_id"] == store][:SAMPLE_N_ITEMS]
sample = (
    series_index.loc[rows, ["store_id", "item_id"]]
    .join(pd.DataFrame(units[rows], index=rows, columns=day_cols))
    .melt(id_vars=["store_id", "item_id"], var_name="d", value_name="units")
)
merged = (
    sample.merge(calendar[["d", "date", "wm_yr_wk"]], on="d", how="left")
    .merge(prices[key + ["sell_price"]], on=key, how="left")
)
merged["d_num"] = merged["d"].str.removeprefix("d_").astype(int)
merged = merged.sort_values(["item_id", "d_num"]).reset_index(drop=True)
print(f"filas: {len(merged):,}")
display(profile_frame(merged))
merged.head(10)

filas: 5,823


,dtype,nulls,null_pct,unique
store_id,str,0,0.0,1
item_id,str,0,0.0,3
d,str,0,0.0,1941
units,float32,0,0.0,13
date,str,0,0.0,1941
wm_yr_wk,int64,0,0.0,278
sell_price,float64,0,0.0,7
d_num,int64,0,0.0,1941


,store_id,item_id,d,units,date,wm_yr_wk,sell_price,d_num
0,CA_1,FOODS_1_001,d_1,3.0,2011-01-29,11101,2.0,1
1,CA_1,FOODS_1_001,d_2,0.0,2011-01-30,11101,2.0,2
2,CA_1,FOODS_1_001,d_3,0.0,2011-01-31,11101,2.0,3
3,CA_1,FOODS_1_001,d_4,1.0,2011-02-01,11101,2.0,4
4,CA_1,FOODS_1_001,d_5,4.0,2011-02-02,11101,2.0,5
5,CA_1,FOODS_1_001,d_6,2.0,2011-02-03,11101,2.0,6
6,CA_1,FOODS_1_001,d_7,0.0,2011-02-04,11101,2.0,7
7,CA_1,FOODS_1_001,d_8,2.0,2011-02-05,11102,2.0,8
8,CA_1,FOODS_1_001,d_9,0.0,2011-02-06,11102,2.0,9
9,CA_1,FOODS_1_001,d_10,0.0,2011-02-07,11102,2.0,10


## 5. Máscaras

Se aplican las funciones de la etapa 1 sobre los arreglos completos:

- `build_daily_prices`: precio de cada tienda-artículo-día (NaN si la semana no tiene precio).
- `compute_launch`: lanzamiento = primera semana con precio.
- `apply_masks`: unidades y precios enmascarados, registro de rechazos y `mask_reason` (motivo de
  cada celda).

In [18]:
calendar

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1964,2016-06-15,11620,Wednesday,5,6,2016,d_1965,NaN,NaN,NaN,NaN,0,1,1
1965,2016-06-16,11620,Thursday,6,6,2016,d_1966,NaN,NaN,NaN,NaN,0,0,0
1966,2016-06-17,11620,Friday,7,6,2016,d_1967,NaN,NaN,NaN,NaN,0,0,0
1967,2016-06-18,11621,Saturday,1,6,2016,d_1968,NaN,NaN,NaN,NaN,0,0,0


In [19]:
prices

,store_id,item_id,wm_yr_wk,sell_price,csv_row
0,CA_1,HOBBIES_1_001,11325,9.58,0
1,CA_1,HOBBIES_1_001,11326,9.58,1
2,CA_1,HOBBIES_1_001,11327,8.26,2
3,CA_1,HOBBIES_1_001,11328,8.26,3
4,CA_1,HOBBIES_1_001,11329,8.26,4
...,...,...,...,...,...
6841116,WI_3,FOODS_3_827,11617,1.00,6841116
6841117,WI_3,FOODS_3_827,11618,1.00,6841117
6841118,WI_3,FOODS_3_827,11619,1.00,6841118
6841119,WI_3,FOODS_3_827,11620,1.00,6841119


In [20]:
series_index

,store_id,item_id,csv_row
0,CA_1,FOODS_1_001,1612
1,CA_1,FOODS_1_002,1613
2,CA_1,FOODS_1_003,1614
3,CA_1,FOODS_1_004,1615
4,CA_1,FOODS_1_005,1616
...,...,...,...
30485,WI_3,HOUSEHOLD_2_512,29048
30486,WI_3,HOUSEHOLD_2_513,29049
30487,WI_3,HOUSEHOLD_2_514,29050
30488,WI_3,HOUSEHOLD_2_515,29051


In [21]:
wod = week_of_day(calendar, LAST_DAY)
prices_daily = build_daily_prices(prices, wod, series_index)
launch = compute_launch(prices_daily, series_index, wod)


In [22]:
wod

array([11101, 11101, 11101, ..., 11616, 11617, 11617], shape=(1941,))

In [23]:
prices_daily

array([[ 2.  ,  2.  ,  2.  , ...,  2.24,  2.24,  2.24],
       [ 7.88,  7.88,  7.88, ...,  9.48,  9.48,  9.48],
       [ 2.88,  2.88,  2.88, ...,  3.23,  3.23,  3.23],
       ...,
       [18.47, 18.47, 18.47, ..., 19.54, 19.54, 19.54],
       [  nan,   nan,   nan, ...,  1.97,  1.97,  1.97],
       [ 5.94,  5.94,  5.94, ...,  5.94,  5.94,  5.94]],
      shape=(30490, 1941), dtype=float32)

In [24]:
launch

,store_id,item_id,launch_wk,launch_d
0,CA_1,FOODS_1_001,11101.0,1.0
1,CA_1,FOODS_1_002,11101.0,1.0
2,CA_1,FOODS_1_003,11101.0,1.0
3,CA_1,FOODS_1_004,11206.0,400.0
4,CA_1,FOODS_1_005,11101.0,1.0
...,...,...,...,...
30485,WI_3,HOUSEHOLD_2_512,11101.0,1.0
30486,WI_3,HOUSEHOLD_2_513,11311.0,799.0
30487,WI_3,HOUSEHOLD_2_514,11101.0,1.0
30488,WI_3,HOUSEHOLD_2_515,11352.0,1086.0


In [25]:
raise Exception("Detener")

Exception: Detener

In [ ]:

# units_m, prices_m, rejections, mask_reason = apply_masks(
#     units, prices_daily, launch, series_index, wod, prices, cfg["data"]["sales_file"]
# )
# pd.Series(MASK_REASONS, name="motivo").rename_axis("código").to_frame()

In [ ]:
# raise Exception("Detener")

La tabla unida con los valores originales (`units`, `sell_price`) y, al lado, los
enmascarados (`units_masked`, `price_masked`) y el motivo (`mask_reason`). Un NaN en las columnas
enmascaradas significa "excluido", no cero.

In [ ]:
labels = np.array(list(MASK_REASONS.values()))
pos = merged["d_num"].to_numpy() - 1
row_of = pd.Series(rows, index=series_index.loc[rows, "item_id"]).reindex(merged["item_id"]).to_numpy()
merged["units_masked"] = units_m[row_of, pos]
merged["price_masked"] = prices_m[row_of, pos]
merged["mask_reason"] = labels[mask_reason[row_of, pos]]
merged = merged.merge(launch[["store_id", "item_id", "launch_d"]], on=["store_id", "item_id"])
merged[["item_id", "d", "date", "wm_yr_wk", "launch_d", "units", "units_masked", "sell_price",
        "price_masked", "mask_reason"]].head(15)

In [ ]:
# Solo las filas que cambian en la muestra.
merged.loc[merged["mask_reason"] != "valid",
           ["item_id", "d", "units", "units_masked", "sell_price", "mask_reason"]]

## 6. Impacto de las máscaras

Sobre todas las celdas (series × días): cuántas quedan válidas y por qué se excluye el resto.
`pre_launch` no es un error (el artículo aún no existía); los rechazos son `sales_before_first_price`,
`invalid_units` e `invalid_price_with_sales`.

In [ ]:
counts = pd.Series(np.bincount(mask_reason.ravel(), minlength=len(MASK_REASONS)),
                   index=labels, name="celdas")
# Valor absoluto: una unidad negativa también es volumen que se excluye.
raw = np.abs(np.nan_to_num(units))
units_by_reason = pd.Series(
    [float(raw[mask_reason == c].sum()) for c in MASK_REASONS], index=labels, name="unidades")
impact = pd.DataFrame({
    "celdas": counts,
    "% celdas": (counts / counts.sum() * 100).round(3),
    "|unidades crudas|": units_by_reason,
    "% unidades": (units_by_reason / units_by_reason.sum() * 100).round(3),
    "rechazo": [c in REJECTION_CODES for c in MASK_REASONS],
})
impact

In [ ]:
group = np.where(mask_reason == 0, "válida",
                 np.where(np.isin(mask_reason, REJECTION_CODES), "rechazo", "excluida (no error)"))
shares = pd.Series(group.ravel()).value_counts(normalize=True).mul(100).round(3)
shares.rename("% celdas").to_frame()

In [ ]:
# Por tienda y por categoría (prefijo de item_id): % de celdas válidas y de rechazo.
cat = series_index["item_id"].str.split("_").str[0]
per_series = pd.DataFrame({
    "store_id": series_index["store_id"],
    "cat_id": cat,
    "válidas": (mask_reason == 0).sum(axis=1),
    "rechazos": np.isin(mask_reason, REJECTION_CODES).sum(axis=1),
    "celdas": mask_reason.shape[1],
})
for by in ("store_id", "cat_id"):
    g = per_series.groupby(by)[["válidas", "rechazos", "celdas"]].sum()
    display((g[["válidas", "rechazos"]].div(g["celdas"], axis=0) * 100).round(2).add_prefix("% "))